SIGNAL TEST

The goal of this section is to add a signal component to the model and get an estimate for the posterior distribution p(mu_s|data). After that we shall proceed with model comparison. 

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
from scipy.integrate import trapezoid

THE CODE BELOW IS A RELOAD OF ALL THE IMPORTANT VARIABLES

In [3]:
bins = pd.read_csv('SourceData/s2_binning_info.csv')
resp_nr = pd.read_csv('SourceData/s2_response_nr.csv')
resp_er = pd.read_csv('SourceData/s2_response_er.csv')
bg = pd.read_csv('SourceData/er_and_cevns_background.csv')
events = pd.read_csv('SourceData/events_after_cuts.csv')

In [4]:
s2_bin_centers_log = bins['log_center_pe'].values
s2_bin_centers_lin = bins['linear_center_pe'].values
s2_bin_widths = (bins['end_pe'] - bins['start_pe']).values
s2_bin_edges = np.concatenate([bins['start_pe'].values, [bins['end_pe'].iloc[-1]]])

In [5]:
#Exctracting information from nuclear recoil response
s2_energies = resp_nr['energy_kev'].values
bin_starts = resp_nr['energy_bin_start_kev'].values
bin_ends = resp_nr['energy_bin_end_kev'].values
dE = bin_ends - bin_starts

response_matrix_nr = resp_nr.values[:,3:] #we start from the 4th column, since the 3 previous ones are energies. The 4th column is s2_bin_000.

In [6]:
import wimprates as wr
reference_cross_section = 1e-45  # cm^2
rate_pertonneyearkev = wr.rate_wimp_std(
    es=s2_energies, 
    mw=4, 
    sigma_nucleon=reference_cross_section)

c:\Users\ana\DM1StatsGroupRepo2026\venv\Lib\site-packages\wimprates\__init__.py:6: UserWarning: Default WIMP parameters are changed in accordance with https://arxiv.org/abs/2105.00599 (github.com/JelleAalbers/wimprates/pull/14)
  warnings.warn(
c:\Users\ana\DM1StatsGroupRepo2026\venv\Lib\site-packages\wimprates\utils.py:40: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(f)


In [7]:
#Most of the code below was copied directly from the Public Data Release
rate_pertonneyear = rate_pertonneyearkev * dE
rate_before_cutoff = rate_pertonneyear * 0.97678 
#The authors of the paper remove events below 0.7keV
recoil_energy_cutoff_kev = 0.7
rate_after_cutoff = rate_before_cutoff.copy()

# Which bin contains the cutoff?
cutoff_bin_index = (bin_starts < recoil_energy_cutoff_kev).sum() - 1 #This counts how many bins start below 0.7 keV, then subtracts 1 to get the index

# All bins fully below 0.7 keV are removed
rate_after_cutoff[:cutoff_bin_index] = 0

# Suppress the spectrum proportionally in the bin with the cutoff
suppress_by = (
    (recoil_energy_cutoff_kev - bin_starts[cutoff_bin_index]) 
    / bin_starts[cutoff_bin_index])
assert 0 <= suppress_by <= 1

#Only keep the part above the cutoff
rate_after_cutoff[cutoff_bin_index] *= 1 - suppress_by 

In [8]:
b_er = bg['er_background_events']
b_cevns = bg['cevns_background_events']
b_nominal = b_er + b_cevns
k_obs, _ = np.histogram(events['s2_area_pe'].values, bins=s2_bin_edges)

s_i = rate_after_cutoff @ response_matrix_nr
b_i = b_nominal.values

Now we add a signal to the model

We define the joint likelihood function $\mathcal{L}(\text{data} \mid \mu_s, \beta)$ to account for both the dark matter signal strength and the background normalization. However, now we use Bayesian Inference, which treats both the signal strength $\mu_s$ and background normalization $\beta$ as random variables.

We are using the same Region of Interest used for the Frequentist Likelihood (which treated $\mu_s$ as a fixed parameter).

In [ ]:
# Reusing the ROI
roi = (165.3, 271.7)
roi_mask = (s2_bin_centers_log >= roi[0]) & (s2_bin_centers_log <= roi[1])

# Redefine μ_i to accept both μ_s and β as arguments
def mu_i_combined(mu_s, beta):
    """
    Expected events per bin, using the ROI mask defined in the likelihood file.
    """
    return beta * b_nominal.values[roi_mask] + mu_s * s_i[roi_mask]

# Log-likelihood
def log_likelihood_combined(mu_s, beta):
    """
    Joint log-likelihood for signal strength (μ_s) and background normalization (β).
    """
    expected = mu_i_combined(mu_s, beta)
    expected = np.clip(expected, 1e-12, None)
    return np.sum(stats.poisson.logpmf(k_obs[roi_mask], expected))

# Likelihood (L = exp(logL)) for integration
def likelihood_combined(mu_s, beta):
    return np.exp(log_likelihood_combined(mu_s, beta))

BAYES FACTOR

Now that we have a likelihood that is a function of both $\mu_s$ and $\beta$, we can calculate the Bayes factor. For that, we need to calculate the evidence of model 0 (only background) $P(data \mid H_0)$, and of model 1 (background+signal) $P(data \mid H_1)$.

We will calculate the Bayes factor for two different priors: flat (uniform) prior and scale invariant prior (both normalized). In this way, we will study the sensitivity of Bayes factor to prior.

We are starting with a flat prior, and the evidence can be calculated as the integral of the likelihood multiplied by the prior over the parameter space.

In [22]:
# Range definitions
mu_s_min, mu_s_max = 0.001, 100.0  # Start at 0.001 to avoid log(0)
beta_min, beta_max = 0.1, 30.0

mu_s_values = np.linspace(mu_s_min, mu_s_max, 100)
beta_values = np.linspace(beta_min, beta_max, 100)

# ---------------------------------------------------------
# FLAT PRIORS
# ---------------------------------------------------------
# Normalization constants (Area=1)
c_flat_beta = 1.0 / (beta_max - beta_min)
c_flat_mu = 1.0 / (mu_s_max - mu_s_min)

# P(data | H0)_flat (evidence for model 0: mu_s = 0)
likes_H0_flat = np.array([likelihood_combined(0, b) * c_flat_beta for b in beta_values])
P_data_H0_flat = trapezoid(likes_H0_flat, beta_values)

# P(data | H1)_flat (evidence for model 1)
evidence_grid_H1_flat = np.zeros((len(mu_s_values), len(beta_values)))
for i, m in enumerate(mu_s_values):
    for j, b in enumerate(beta_values):
        evidence_grid_H1_flat[i, j] = likelihood_combined(m, b) * c_flat_beta * c_flat_mu

P_data_H1_flat = trapezoid(trapezoid(evidence_grid_H1_flat, beta_values, axis=1), mu_s_values)

# Bayes factor
BF_01_flat = P_data_H0_flat / P_data_H1_flat
print (f"Bayes Factor (Flat Priors): {BF_01_flat:.4f}")

Bayes Factor (Flat Priors): 0.8881


Then, for the scale invariant prior:

In [23]:
# ---------------------------------------------------------
# SCALE INVARIANT PRIORS
# ---------------------------------------------------------
# Normalization constants (Integral of 1/x is ln(x)))
# Integral from a to b of (1/x) dx = ln(b) - ln(a)
c_si_beta = 1.0 / (np.log(beta_max) - np.log(beta_min))
c_si_mu = 1.0 / (np.log(mu_s_max) - np.log(mu_s_min))

def pi_si_beta(b): return c_si_beta / b
def pi_si_mu(m): return c_si_mu / (m)

# P(data | H0)_SI (evidence for model 0: mu_s = 0)
likes_H0_SI = np.array([likelihood_combined(0, b) * pi_si_beta(b) for b in beta_values])
P_data_H0_SI = trapezoid(likes_H0_SI, beta_values)

# P(data | H1)_SI (evidence for model 1)
evidence_grid_H1_SI = np.zeros((len(mu_s_values), len(beta_values)))
for i, m in enumerate(mu_s_values):
    for j, b in enumerate(beta_values):
        evidence_grid_H1_SI[i, j] = likelihood_combined(m, b) * pi_si_mu(m) * pi_si_beta(b)

P_data_H1_SI = trapezoid(trapezoid(evidence_grid_H1_SI, beta_values, axis=1), mu_s_values)

# Bayes factor
BF_01_SI = P_data_H0_SI / P_data_H1_SI
print(f"Bayes Factor (Scale Invariant): {BF_01_SI:.4f}")

Bayes Factor (Scale Invariant): 0.0225


We can compare both values of the Bayes factor in function of the prior applied:

In [25]:
# Final Comparison
print(f"Bayes Factor (Flat Priors): {BF_01_flat:.4f}")
print(f"Bayes Factor (Scale Invariant): {BF_01_SI:.4f}")

Bayes Factor (Flat Priors): 0.8881
Bayes Factor (Scale Invariant): 0.0225


As we can see, we obtained quite different results for each prior, but they both have one thing in common: they are lower than 1. This impplies that the data observed is more likely under the signal hypothesis ($H_1$) than the background-only hypothesis ($H_0$).

Regarding to the prior sensitivity, we can conclude that the evidences of the models (and therefore the Bayes factor) are quite sensitive to the type of prior used. This is because the flat prior penalizes the signal model significantly by spreading probability mass over a large, unsupported range of signal strengths. It states that the signal being 1 is as likely as it being 99; however, the excess of events is actually in smaller values, so the flat prior is not fair with the signal values (gives more importance to higher values than it should). On the other hand, the scale invariant prior is more appropriate when the order of magnitude of the signal is unknown.  It gives equal weight to every decade (e.g., 0.1 to 1 and 1 to 10), making it more "neutral". Consequently, it is more fair with small values of the signal, and it notice that the model $H_1$ fits pretty good the events observed. That is why the scale invariant prior favors much more the model with the dark matter signal over the background-only model.

However, the dramatically low value for $B_{01}$ in the Scale invariant prior case could indicate us that our background model is too rigid. The lack of flexibility could prevent the background from adapting to spectral fluctuations, forcing the Bayesian evidence to favor the model with the dark matter signal just because the background-only model can not properly adapt to the data.